In [1]:
import pandas as pd
import joblib
import numpy as np


interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

metadata = joblib.load("data\\11092026\\metadata.pkl")

interactions

,user_idx,movie_idx,rating,timestamp
0,0,1104,5.0,978300760
1,0,639,3.0,978302109
2,0,853,3.0,978301968
3,0,3177,4.0,978300275
4,0,2162,5.0,978824291
...,...,...,...,...
1000204,6039,1019,1.0,956716541
1000205,6039,1022,5.0,956704887
1000206,6039,548,5.0,956704746
1000207,6039,1024,4.0,956715648


In [2]:
n_movies = metadata["n_movies"]
n_users = metadata["n_users"]

### Methods for matrix factorization

In [4]:
def train_(train, n_users, n_movies, n_factor, lr, reg, epochs=20, show=False, random_state=42):

    rng = np.random.default_rng(random_state)

    P = rng.normal(
        0,
        0.1,
        size=(n_users, n_factor),
    )

    Q = rng.normal(
        0,
        0.1,
        size=(n_movies, n_factor),
    )

    mu = train.rating.mean()

    bu = np.zeros(n_users)
    bi = np.zeros(n_movies)

    for epoch in range(epochs):
        errors = []

        for row in train.itertuples(index=False):

            u = row.user_idx
            m = row.movie_idx
            r = row.rating

            prediction = (
                mu + bu[u] + bi[m] + P[u] @ Q[m]
            )

            error = r - prediction

            errors.append(error)

            old_p = P[u].copy()

            bu[u] += lr * (
                error - reg * bu[u]
            )

            bi[m] += lr * (
                error - reg * bi[m]
            )

            P[u] += lr * (
                error * Q[m] - reg * P[u]
            )

            Q[m] += lr * (
                error * old_p - reg * Q[m]
            )

        if show:
            rmse = np.sqrt(
                np.mean(
                    np.square(errors)
                )
            )
            print(
                f"Epoch {epoch + 1}: "
                f"RMSE {rmse:.4}: "
            )

    return P, Q, mu, bu, bi

In [5]:
def recommend(df, user_idx, P, Q, mu, bu, bi, k=10):

    watched = (
        df.loc[
            df.user_idx == user_idx,
            "movie_idx"
        ]
    ).unique()

    scores = (
        mu + bu[user_idx] + bi + P[user_idx] @ Q.T
        )

    scores[watched] = -np.inf

    recomemend = np.argsort(scores)[::-1][:k]

    return [
        (movie_idx, scores[movie_idx])
        for movie_idx in recomemend
    ]


In [6]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)

def metrics(P, Q, mu, bu, bi, train, val, k=10):

    recalls = []
    precisions = []
    ndcgs = []

    for user_idx in val.user_idx.unique():

        recommendations = recommend(train, user_idx, P, Q, mu, bu, bi, k)
        val_user = val[val.user_idx == user_idx]

        recommended = [
            movie_idx
            for movie_idx, _ in recommendations
        ]

        relevant = val_user.loc[
            val_user.rating >= 4,
            "movie_idx"
        ].tolist()

        if len(relevant) == 0:
            continue

        recalls.append(recall_at_k(relevant, recommended, k))
        precisions.append(precision_at_k(relevant, recommended, k))
        ndcgs.append(ndcg_at_k(relevant, recommended, k))

    return (
        np.mean(recalls),
        np.mean(precisions),
        np.mean(ndcgs)
    )

### Grid Search


In [7]:
from itertools import product

param_grid = {
    "n_factors": [5, 10, 20, 40],
    "lr": [0.005, 0.01, 0.02],
    "reg": [0.01, 0.02, 0.05],
}

In [8]:
def grid_search(param_grid, train, val, n_users, n_movies, show):

    res = []

    i = 0

    for n_factor, lr, reg in product(
        param_grid["n_factors"],
        param_grid["lr"],
        param_grid["reg"]
    ):
        i += 1

        print(f"attempt: {i}")
        print(f"n_factors={n_factor}, lr={lr}, reg={reg}")

        P, Q, mu, bu, bi = train_(train, n_users, n_movies, n_factor, lr, reg, show=show)

        recall, precision, ndcg = metrics(P, Q, mu, bu, bi, train, val)
        
        res.append({
            "n_factors": n_factor,
            "lr": lr,
            "reg": reg,
            "recall": recall,
            "precision": precision,
            "ndcg": ndcg
        })

    return res


In [ ]:
def grid_epoch(train, val, n_users, n_movies, n_factors, lr, reg, epochs, show):

    res = []

    for epoch in epochs:

        print(f"epoch: {epoch}")

        P, Q, mu, bu, bi = train_(train, n_users, n_movies, n_factors, lr, reg, epoch, show)
        
        recall, precision, ndcg = metrics(P, Q, mu, bu, bi, train, val)

        res.append({
            "epoch": epoch,
            "recall": recall,
            "precision": precision,
            "ndcg": ndcg
        })

    return res

### Search params for factorization

In [10]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit

train_parts, val_parts, test_parts = TemporalSplit().split(data=interactions)

print(train_parts.shape)
print(test_parts.shape)
print(val_parts.shape)

(797758, 4)
(105732, 4)
(96719, 4)


In [ ]:
res = grid_search(param_grid, train_parts, val_parts, n_users, n_movies, False)

In [25]:
res_df = pd.DataFrame(res)
res_df

,n_factors,lr,reg,recall,precision,ndcg
0,5,0.005,0.01,0.024189,0.023540,0.027324
1,5,0.005,0.02,0.022222,0.022036,0.024604
2,5,0.005,0.05,0.016818,0.016859,0.018747
3,5,0.010,0.01,0.028469,0.024432,0.031123
4,5,0.010,0.02,0.026635,0.022840,0.028901
5,5,0.010,0.05,0.020714,0.018730,0.022368
6,5,0.020,0.01,0.022801,0.018346,0.023801
7,5,0.020,0.02,0.021752,0.017419,0.022407
8,5,0.020,0.05,0.020594,0.015722,0.019676
9,10,0.005,0.01,0.024971,0.024851,0.030508


In [26]:
res_df.to_parquet(
    "data\\matrix_factorization\\02_parametres.parquet",
    index= False
)


In [27]:
n_factors = 10
lr = 0.01
reg = 0.01

In [ ]:
epochs = [5, 10, 20, 40, 80]

res = grid_epoch(train_parts, val_parts, n_users, n_movies, n_factors, lr, reg, epochs, False)

In [13]:
res_df = pd.DataFrame(res)
res_df

,epoch,recall,precision,ndcg
0,5,0.015222,0.015407,0.018905
1,10,0.024215,0.023697,0.028839
2,20,0.027075,0.024362,0.031765
3,40,0.021281,0.018608,0.023021
4,80,0.013618,0.012172,0.013895


In [14]:
res_df.to_parquet(
    "data\\matrix_factorization\\03_parametres.parquet",
    index= False
)


Из подобранных параметров оптимальными оказались:
<center>n_factors = 10 </center>
<center>lr = 0.01 </center>
<center>reg = 0.01 </center>
<center>epochs = 20 </center>

### Final 

Финальное обучение факторизации

In [15]:
n_factors = 10
lr = 0.01
reg = 0.01
epochs = 20

In [ ]:
train_parts, val_parts, test_parts = TemporalSplit().split(data=interactions)

P, Q, mu, bu, bi = train_(train_parts, n_users, n_movies, n_factors, lr, reg, epochs, True)

recall, precision, ndcg = metrics(P, Q, mu, bu, bi, train_parts, val_parts)


Epoch 1: RMSE 0.9481: 
Epoch 2: RMSE 0.9101: 
Epoch 3: RMSE 0.9019: 
Epoch 4: RMSE 0.8966: 
Epoch 5: RMSE 0.8894: 
Epoch 6: RMSE 0.8785: 
Epoch 7: RMSE 0.8658: 
Epoch 8: RMSE 0.8536: 
Epoch 9: RMSE 0.8426: 
Epoch 10: RMSE 0.833: 
Epoch 11: RMSE 0.8248: 
Epoch 12: RMSE 0.8178: 
Epoch 13: RMSE 0.8118: 
Epoch 14: RMSE 0.8067: 
Epoch 15: RMSE 0.8022: 
Epoch 16: RMSE 0.7984: 
Epoch 17: RMSE 0.795: 
Epoch 18: RMSE 0.792: 
Epoch 19: RMSE 0.7893: 
Epoch 20: RMSE 0.787: 


In [19]:
model = {
    "P": P,
    "Q": Q,
    "mu": mu,
    "bu": bu,
    "bi": bi,
    "parameters": {
        "n_factors": n_factors,
        "learning_rate": lr,
        "regularization": reg,
        "epochs": epochs,
        "random_state": 42
    },
    "metrics": {
        "recall@10": recall,
        "precision@10": precision,
        "ndcg@10": ndcg
    }
}

In [21]:
joblib.dump(
    model,
    "data\\models\\matrix_factorization.pkl"
)

['data\\models\\matrix_factorization.pkl']